# Part 2 — free Qwen inference with saved Roboflow browser results

This notebook runs Qwen on all seven photographs and reuses the Roboflow results already obtained through the free **Try This Model** page. **No API key or paid API is required.** The Roboflow model is not rerun here.

1. Choose **Runtime → Change runtime type → T4 GPU**, subject to free availability. Do not buy compute units.
2. Run cells in order. Qwen2.5-VL-3B downloads several GB and runs on the Colab GPU.
3. Keep the downloaded results ZIP and download the executed notebook using **File → Download → Download .ipynb**.
4. Results clearly distinguish newly executed Qwen detections from saved browser Roboflow detections. No accuracy claim is established without reference annotations.

Sources: [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct), [Roboflow model by twar](https://universe.roboflow.com/twar/empty-shelf-detection-r8cfz), CC BY 4.0.

Numerical-stability update: Qwen uses float32, eager attention and smaller image inputs. Automatic CPU offloading limits GPU memory use; generation may be slower. This change still needs execution in Colab.


In [1]:
import subprocess, sys
packages = [
    'transformers==4.57.1', 'accelerate>=0.34,<2',
    'qwen-vl-utils==0.0.14', 'Pillow', 'requests'
]
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', *packages],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
print(result.stdout)
result.check_returncode()
print('Dependencies installed. If Colab requests a runtime restart, restart and continue with the next cell.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 46.2 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.21.0
    Uninstalling transformers-5.21.0:
      Successfully uninstalled transformers-5.21.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following d

In [2]:
import io, json, zipfile, shutil, hashlib, datetime, importlib.metadata
from pathlib import Path
import requests
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime → Change runtime type → T4 GPU, then run again. Do not buy compute units.')
ROOT=Path('/content/shelf_inference_assignment')
ROOT.mkdir(exist_ok=True)
url='https://codeload.github.com/MVEST22/-ai-accounting-ml-assignment./zip/refs/heads/main'
r=requests.get(url,timeout=120);r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as archive:
    for info in archive.infolist():
        if '..' in Path(info.filename).parts or Path(info.filename).is_absolute():
            raise ValueError('Unexpected archive path')
    archive.extractall(ROOT)
BASE=next(ROOT.glob('*/Part 2 - shelves'))
IMAGES=sorted(BASE.glob('zone*/*.jpg'))
assert len(IMAGES)==7
OUT=ROOT/'programmatic_results'
OUT.mkdir(exist_ok=True)
(OUT/'raw_llm').mkdir(exist_ok=True)
(OUT/'raw_roboflow').mkdir(exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0))
print('Loaded',len(IMAGES),'photos from the repository.')


GPU: Tesla T4
Loaded 7 photos from the repository.


In [3]:
# Reuse the actual results previously copied from the free Roboflow browser demo.
# This cell makes no inference API request.
saved_path=BASE/'results'/'roboflow_browser.json'
cv=json.loads(saved_path.read_text())
assert cv['model_id']=='empty-shelf-detection-r8cfz/1'
assert cv['confidence_threshold']==0.25 and cv['overlap_threshold']==0.5
assert {i['image'] for i in cv['images']}=={p.relative_to(BASE).as_posix() for p in IMAGES}
for item in cv['images']:
    assert all(b['class']=='Empty-space' and float(b['confidence'])>=0.25 for b in item['predictions'])
    (OUT/'raw_roboflow'/(Path(item['image']).stem+'.json')).write_text(json.dumps(item,indent=2))
    print(Path(item['image']).name[:23],len(item['predictions']),'saved empty areas')
shutil.copyfile(saved_path,OUT/'roboflow_browser.json')
print('Loaded seven saved Roboflow browser results. No API key or new Roboflow inference required.')


IMG-20250318-WA0009_jpg 1 saved empty areas
IMG-20250318-WA0010_jpg 1 saved empty areas
IMG-20250318-WA0011_jpg 1 saved empty areas
IMG-20250318-WA0012_jpg 10 saved empty areas
IMG-20250318-WA0014_jpg 17 saved empty areas
IMG-20250318-WA0017_jpg 11 saved empty areas
IMG-20250318-WA0019_jpg 6 saved empty areas
Loaded seven saved Roboflow browser results. No API key or new Roboflow inference required.


In [4]:
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
MODEL='Qwen/Qwen2.5-VL-3B-Instruct'
import gc
# Release any prior model before retrying this cell.
if 'model' in globals():
    del model
for stale in ['inputs','generated','trimmed']:
    globals().pop(stale,None)
gc.collect()
torch.cuda.empty_cache()
torch.manual_seed(0)
model=Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL,torch_dtype=torch.float32,device_map='auto',
    max_memory={0:'12GiB','cpu':'8GiB'},offload_folder=str(ROOT/'model_offload'),
    attn_implementation='eager')
model.eval()
processor=AutoProcessor.from_pretrained(MODEL,min_pixels=128*28*28,max_pixels=384*28*28)
PROMPT='Identify contiguous visibly empty front-facing shelf areas needing replenishment in this photograph. Count each area once, not individual missing products. Exclude the floor, black image borders, normal shelf edges and ordinary narrow spacing between stocked products. Stock pushed to the back with exposed shelf front may be a replenishment area. Return only a JSON object with a boxes array. For each area return x1, y1, x2, y2 and reason. Coordinates MUST be normalized from 0 to 1000 across the full image: top-left (0,0), bottom-right (1000,1000). Each box must have x1<x2 and y1<y2. If there are no empty areas return {"boxes": []}. Do not copy or refer to any other model results.'
(OUT/'llm_prompt.txt').write_text(PROMPT)
def parse_qwen_boxes(response):
    import math
    payload=response.strip()
    if payload.startswith('```'):
        lines=payload.splitlines()
        if lines[-1].strip()!='```':
            raise ValueError('Unclosed JSON fence')
        payload='\n'.join(lines[1:-1]).strip()
    parsed=json.loads(payload)
    normalized=[]
    def visit(value, inherited_reason=''):
        if isinstance(value, dict):
            reason=str(value.get('reason',inherited_reason))
            if all(k in value for k in ('x1','y1','x2','y2')):
                visit([value[k] for k in ('x1','y1','x2','y2')],reason)
            elif 'boxes' in value:
                visit(value['boxes'],reason)
            else:
                raise ValueError('Unknown box object')
        elif isinstance(value,list):
            if len(value)==4 and all(isinstance(v,(int,float)) and not isinstance(v,bool) for v in value):
                x1,y1,x2,y2=map(float,value)
                if not all(math.isfinite(v) for v in (x1,y1,x2,y2)):
                    raise ValueError('Nonfinite coordinates')
                if not (0<=x1<x2<=1000 and 0<=y1<y2<=1000):
                    raise ValueError('Invalid normalized coordinates')
                normalized.append({'x1':x1*.64,'y1':y1*.64,'x2':x2*.64,'y2':y2*.64,'reason':inherited_reason})
            else:
                for item in value:
                    visit(item,inherited_reason)
        else:
            raise ValueError('Unsupported box format')
    visit(parsed)
    return normalized

llm={'source':'Programmatic Qwen visual inference executed in Colab; no other model outputs included in prompts',
     'model_id':MODEL,'unit':'contiguous empty front-facing shelf areas, not missing product units','images':[]}
for path in IMAGES:
    messages=[{'role':'user','content':[{'type':'image','image':Image.open(path).convert('RGB')},{'type':'text','text':PROMPT}]}]
    text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    image_inputs,video_inputs=process_vision_info(messages)
    inputs=processor(text=[text],images=image_inputs,videos=video_inputs,padding=True,return_tensors='pt').to('cuda')
    with torch.inference_mode():
        generated=model.generate(**inputs,max_new_tokens=2048,do_sample=False)
    trimmed=[out[len(inp):] for inp,out in zip(inputs.input_ids,generated)]
    response=processor.batch_decode(trimmed,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
    (OUT/'raw_llm'/(path.stem+'.txt')).write_text(response)
    if not response.strip() or set(response.strip())=={'!'}:
        raise RuntimeError('Qwen produced degenerate output; raw response saved. Do not treat this as a detection result.')
    try:
        boxes=parse_qwen_boxes(response)
    except (ValueError,KeyError,TypeError) as error:
        print('Raw Qwen response:',response)
        raise RuntimeError('Invalid Qwen response for '+path.name+': '+str(error)+'. Raw response saved; no count substituted.') from error
    llm['images'].append({'image':path.relative_to(BASE).as_posix(),'width':640,'height':640,'boxes':boxes})
    (OUT/'llm_programmatic.json').write_text(json.dumps(llm,indent=2))
    print(path.name[:23],len(boxes),'empty areas')
    del inputs,generated,trimmed
    torch.cuda.empty_cache()
print('Qwen has now run on all seven photographs; saved Roboflow browser results are retained for comparison.')


`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


IMG-20250318-WA0009_jpg 1 empty areas
IMG-20250318-WA0010_jpg 1 empty areas
IMG-20250318-WA0011_jpg 1 empty areas
IMG-20250318-WA0012_jpg 1 empty areas
IMG-20250318-WA0014_jpg 2 empty areas
IMG-20250318-WA0017_jpg 1 empty areas
IMG-20250318-WA0019_jpg 1 empty areas
Qwen has now run on all seven photographs; saved Roboflow browser results are retained for comparison.


In [6]:
import csv, base64
assert len(llm['images'])==len(cv['images'])==7
lmap={i['image']:i for i in llm['images']};cmap={i['image']:i for i in cv['images']}
data=[]
for path in IMAGES:
    name=path.relative_to(BASE).as_posix()
    right=[]
    for b in cmap[name]['predictions']:
        x1=b['x']-b['width']/2;y1=b['y']-b['height']/2
        x2=x1+b['width'];y2=y1+b['height']
        assert 0<=x1<x2<=640 and 0<=y1<y2<=640
        right.append({'x1':x1,'y1':y1,'x2':x2,'y2':y2,'confidence':b['confidence']})
    data.append({'image':name,'image_data':'data:image/jpeg;base64,'+base64.b64encode(path.read_bytes()).decode(),
                 'zone':path.parent.name,'name':path.name.split('WA')[1][:4],'llm':lmap[name]['boxes'],'cv':right})
with (OUT/'comparison_programmatic.csv').open('w',newline='') as f:
    w=csv.writer(f);w.writerow(['zone','filename','qwen_llm_empty_areas','roboflow_empty_areas'])
    for i in data:w.writerow([i['zone'],Path(i['image']).name,len(i['llm']),len(i['cv'])])
html=(BASE/'comparison_template.html').read_text()
html=html.replace('Interactive ChatGPT visual estimates','Qwen2.5-VL-3B · actual programmatic run').replace('LLM · ChatGPT','LLM · Qwen2.5-VL')
html=html.replace('Browser outputs supplied by the user; no hosted API call was used to produce these saved results. LLM boxes were created interactively after the specialized outputs were available, so this is not a blinded accuracy experiment.',
 'Qwen executed programmatically in Colab. Roboflow detections were reused from the earlier free browser demo and were not rerun. Qwen prompts contained no specialized detections. No ground truth was supplied; this is not a validated accuracy measurement.')
# The recommendation must be reassessed against the newly generated detections.
start=html.index('<article><h2>Recommendation for store employees')
end=html.index('</article>',start)+len('</article>')
html=html[:start]+'<article><h2>Recommendation pending visual review</h2><p>Compare both overlays against each original photo before selecting results for employees. Counts are empty areas, not exact missing-unit quantities. Confirm alerts manually and do not assume the higher count is more accurate.</p></article>'+html[end:]
# Remove comments specific to the earlier interactive detections.
html=html.replace("if(i.name==='0009')article.append(element('p','Same count, different locations: compare the LLM’s central milk-carton gap with the specialized box near the bottom right.'));",'')
html=html.replace("if(i.name==='0011')article.append(element('p','Inspect the LLM’s gap on the second shelf against the specialized box lower down. Neither is a verified reference annotation.'));",'')
html=html.replace('/*DATA*/[]',json.dumps(data).replace('<','\\u003c'))
(OUT/'comparison_programmatic.html').write_text(html)
manifest={'executed_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'llm_model':MODEL,'roboflow_model':cv['model_id'],'roboflow_source':cv['source'],'roboflow_rerun':False,'confidence_threshold':.25,'overlap_threshold':.5,
 'packages':{n:importlib.metadata.version(n) for n in ['torch','transformers','qwen-vl-utils','requests']},
 'images_sha256':{p.relative_to(BASE).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in IMAGES}}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
shutil.copyfile(BASE/'Part_2_Programmatic_Inference.ipynb',OUT/'inference_source_unexecuted_template.ipynb')
(OUT/'README.txt').write_text('New Qwen programmatic results compared with saved free Roboflow browser results (not rerun). Open comparison_programmatic.html locally and review all boxes. Also download your executed Colab notebook using File > Download > Download .ipynb. The included template notebook is source only and is not execution evidence. Send results ZIP and executed notebook back for final review and recommendation. No API key is included.')
archive=shutil.make_archive(str(ROOT/'part2_programmatic_results'),'zip',OUT)
print('LLM counts:',[len(i['llm']) for i in data])
print('Roboflow counts:',[len(i['cv']) for i in data])
from google.colab import files
files.download(archive)


LLM counts: [1, 1, 1, 1, 2, 1, 1]
Roboflow counts: [1, 1, 1, 10, 17, 11, 6]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>